# COMP219
## Lab 6

So far, we covered some fairly simple models that can be used on relatively small datasets. We learned different approaches for both classification and regression problems.

Today, we will have a look at our first AI model: the __multilayer perceptron__ (MLP; also known as feedforward neural network).

### Multilayer Perceptron: Recap
MLPs are a fundamental type of AI model that can be applied for classification and regression problems. An MLP takes some input data and makes predictions based on this input data. The input data is received through the *input layer* and is then passed through one or more *hidden layers* and the prediction is produced in the *output layer*. Each of these layers have a number of *neurons* (except the input layer; it has nodes rather than neurons). Each component of an MLP is described below.

#### *Neurons and Connections*
Neurons can be found in each layer of the neural network (except the input layer). Each neuron in a layer is connected to all other neurons in the subsequent layer. Each neuron has a bias value assigned to it and each connection has a weight value - these update during the training process. The biases are responsible for transforming the data, while weights indicate the strength of the connection between a pair of neurons.

#### *The Input Layer*
The data is fed into the neural network through the input layer. Since no transformation is taking place at this stage, there are no biases assigned to the nodes (hence they are referred to as nodes rather than neurons).

#### *The Hidden Layer*
The job of each hidden layer is to process the information using the weighted connections described above and a nonlinear activation function. Introducing the nonlinear activation function allows the model to learn complex patterns and relationships within the data.

#### *The Output Layer*
The output layer is responsible for producing the network's prediction based on previous layers' information. Usually, the output layer applies an activation function to convert raw outputs (logits) into meaningful, human-readable information.

#### *How It All Works*
The input layer received the data and passes it to the first hidden layer. A weighted sum of the inputs of the hidden layer is calculated by multiplying the input by the connection weight and then adding the bias term to it.

The weighted sum $ z $ for a neuron in the hidden layer is calculated as:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

Where:
- $ z $ is the weighted sum (or pre-activation) for the neuron.
- $ n $ is the number of inputs to the neuron.
- $ w_i $ is the weight associated with the $ i $-th input.
- $ x_i $ is the $ i $-th input feature from the previous layer.
- $ b $ is the bias term for the neuron.

This result is then passed through an __activation function__ (the type of activation function depends on the task at hand). The activation function applies a threshold and if the weighted sum is above this threshold, the neuron will fire (produce an output). Typically, multiple neurons are expected to fire within the same layer and, therefore, will produce multiple outputs. Each of these are passed on to the next layer's neurons that will then perform the same process (the outputs being fed into a single are combined to form a single input).

The defining characteristics of an MLP architecture are known as *hyperparameters*. We can set these values ourselves. Below is a description of each of the hyperparameters:
- batch size: specifies the number of samples used in one training iteration
- epochs: refers to one complete pass through the entire training data
- number of hidden layers: how many hidden layers the model has
- input nodes and output neurons: the number of input nodes corresponds to the number of features in the dataset; the number of output neurons is equal to the number of classes or predicted values in the task.
- activation function: determines the output of each neuron (applied on the result of weighted sum + bias)
- optimiser: updates connection weights during training
- regulariser: prevents overfitting by introducing constraints or penalties during training
- learning rate: influences the size of steps taken towards the minimum of the loss function

These parameters help us improve model performance, speed up training process, and improve model robustness.

Note: the loss function measures how well the model does. Usually, we use Mean Squared Error or Mean Absolute Error for regression; cross-entropy loss for multi-class, and binary cross-entropy for classification. The loss function is not a hyperparameter.

### Part 1: Constructing an MLP
We will construct our first MLP. The code is based on Xiaowei's script you can find on [GitHub](https://github.com/xiaoweih/AISafetyLectureNotes/blob/main/Part_2/12_FCN.py).

The training process can be described as follows:
1. Feedforward: data is fed into and through the network. As data passes through, the hidden layers, the weighted sums are calculated and activation functions are applied on each neuron.
2. Loss Calculation: the output layer produces a prediction and in this layer, the loss function is used to measure the difference between the predicted output(s) and the actual target value(s), known as the ground truth.
3. Backpropagation: the gradient of the loss with respect to each weight and bias is calculated using the chain rule of calculus. An optimisation algorithm is used to perform *gradient descent*. The update rule is as follows:

$$
w_{i}^{(t+1)} = w_{i}^{(t)} - \eta \cdot \frac{\partial L}{\partial w_{i}}
$$

    Where:

- $ w_{i}^{(t)} $ is the weight of the $ i $-th neuron at iteration $ t $.
- $ w_{i}^{(t+1)} $ is the updated weight after the current iteration.
- $ \eta $ is the learning rate, which controls how much to adjust the weights during training.
- $ \frac{\partial L}{\partial w_{i}} $ is the derivative of the loss function $ L $ with respect to the weight $ w_{i} $, representing how much the loss would change if the weight were adjusted.

4. Iterations and Epochs: the process of feedforward, loss calculation, and backpropagation is repeated many times - the number of epochs only increases when the model has seen the entire dataset in its entirety. We repeat these steps until we reach a pre-defined number of epochs. Remember, we have a batch size, which is less than the size of the training data - an iteration means a single update of weights and biases using the batch size.

We set our batch size to 128 and, for simplicity, we set the number of epochs to 10. We set the learning rate to 0.01.

In [1]:
#setup and training params
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import argparse
import time

#training parameters
args = {
    'batch_size': 128,
    'test_batch_size': 128,
    'epochs': 10,
    'lr': 0.01,
    'no_cuda': False,
    'seed': 1,
    'save_model': False
}

#check if CUDA is available
use_cuda = not args['no_cuda'] and torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

torch.manual_seed(args['seed'])
kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

__The Dataset__: We will use the MNIST dataset for our example. This dataset is a collection of images of handwritten digits.

In [2]:
#load data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = datasets.MNIST('../data', train=True, download=True, transform=transform)
testset = datasets.MNIST('../data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=args['batch_size'], shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(testset, batch_size=args['test_batch_size'], shuffle=False, **kwargs)

__The MLP__: The model architecture we will use consists of an input layer, 3 hidden layers and an output layer.
1. Input layer: each image is of size 28x28 pixels, so we create that many nodes.
2. Hidden layer 1: this layer has 128 neurons
3. Hidden layer 2: this layer has 64 neurons
4. Hidden layer 3: this layer has 32 neurons
5. Output layer: this layer has 10 neurons, one for each digits (0...9)

In the function nn.Linear(in_features, out_features), we set up the input layer by specifying the number of pixels in each image. Then, we specify the number of neurons in the first hidden layer using the out_features parameter. Next time we call the function, we have to have a matching number for the in_features parameter for it to be fully connected.

In the feedforward process, we will use the rectified linear unit (ReLU) activation function. The ReLU function is defined as:

$$
f(z) = \max(0, z)
$$

Where:
- $ f(z) $ is the output of the ReLU function.
- $ z $ is the input to the function (weighted sum + bias).

In simple terms, if the result of the weighted sum + bias is greater than 0, the neuron will fire. 

We are naming this model a victim model - we will attempt to steal it later.

In [3]:
#victim model
#feedforward neural network
class VictimModel(nn.Module):
    def __init__(self):
        super(VictimModel, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128) #input layer, first hidden layer
        self.fc2 = nn.Linear(128, 64) #hidden layer 1 -> hidden layer 2
        self.fc3 = nn.Linear(64, 32) #hidden layer 2 -> hidden layer 3
        self.fc4 = nn.Linear(32, 10) #hidden layer 3 -> output layer

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        return x  #logits

#method for training the victim model
def train_victim_model(model, device, train_loader, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            data = data.view(data.size(0), -1)  #flatten input data
            
            optimizer.zero_grad()  #clear gradients
            output = model(data)  #forward pass
            loss = F.cross_entropy(output, target)  #compute loss
            loss.backward()  #backpropagation
            optimizer.step()  #update weights
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

We can now train our model with all the parameters and architecture set up. We will use stochastic gradient descent for our optimiser. We will save the model for... later use.

In [4]:
#initialise victim model with some optimiser
#SGD = stochastic gradient descent
victim_model = VictimModel().to(device)
optimizer = optim.SGD(victim_model.parameters(), lr=args['lr'])

#train the victim model
print("Training the victim model...")
train_victim_model(victim_model, device, train_loader, optimizer, epochs=args['epochs'])

#save the victim model
torch.save(victim_model.state_dict(), "mnist_trained_model.pth")
print("Victim model saved as 'mnist_trained_model.pth'.")

Training the victim model...
Epoch 1/10, Loss: 1.7764
Epoch 2/10, Loss: 0.6343
Epoch 3/10, Loss: 0.4045
Epoch 4/10, Loss: 0.3355
Epoch 5/10, Loss: 0.2952
Epoch 6/10, Loss: 0.2630
Epoch 7/10, Loss: 0.2366
Epoch 8/10, Loss: 0.2148
Epoch 9/10, Loss: 0.1968
Epoch 10/10, Loss: 0.1816
Victim model saved as 'mnist_trained_model.pth'.


### Part 2: Stealing an MLP
Let's pretend we didn't fit this model. We know nothing about its parameters but we do have access to the data (it is publicly available). We know its general structure and we know what it produces - we know the victim model is an MLP predictions on handwritten digits.

#### The Objective
We want to achieve similar performance to that of the victim model's. Last week, we performed model stealing, but we did not really have to deal with hyperparameters. The task is now more complex. 

First, we initialise a model with randomised hyperparameters. Note that the input layer of the stolen model will consist of 28x28 nodes. In a scenario where we don't have access to the training data, we can take guesses based on general knowledge. Grayscale images tend to be downsized to 28x28 pixels, whereas RGB images tend to be represented as 32x32x3 or 224x224x3 tensors. In a complete black-box scenario, you would need to generate synthetic data that matches the victim model's input dimensions (which is often difficult to achieve and it is beyond the scope of the course).

Let's create a stolen model with random layer sizes.

In [5]:
#stolen model
#feedforward neural network
class StolenModel(nn.Module):
    def __init__(self):
        super(StolenModel, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 1024) 
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 10) 

    def forward(self, x):
        x = x.view(-1, 28 * 28)  #flatten input
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)
        return x

Next, we query the victim so that we can fabricate some training data for ourselves. We can load in the model we exported earlier.

In [6]:
#querying the victim model
def query_victim(victim_model, data_loader, device):
    victim_model.eval()
    inputs = []
    outputs = []

    with torch.no_grad():
        for data, _ in data_loader:
            data = data.to(device)
            data_flat = data.view(data.size(0), -1)
            pred = victim_model(data_flat).cpu()  #get victim model's logits
            inputs.append(data_flat.cpu())  #store the inputs
            outputs.append(pred)  #store the victim's logits

    return torch.cat(inputs), torch.cat(outputs)

In our scenario, we don't know anything about the training data. Hence, we can generate random data (noise) which we can use to query the victim. The victim's output will be used as actual labels to train our surrogate model.

In [7]:
#initialise the victim model
victim_model = VictimModel().to(device)
victim_model.load_state_dict(torch.load('mnist_trained_model.pth'))  #load victim model's weights

#qyery the victim model
print("Querying the victim model...")
train_data, train_labels = query_victim(victim_model, train_loader, device)

Querying the victim model...


We can now decide how we want to train our model.

In [8]:
#train the stolen model using the queried data from the victim
def train_stolen_model(stolen_model, device, train_data, train_labels, epochs, learning_rate=0.001):
    stolen_model.train()
    optimizer = optim.SGD(stolen_model.parameters(), lr=learning_rate)
    
    #create a dataset and data loader from the stolen dataset
    dataset = torch.utils.data.TensorDataset(train_data, torch.argmax(train_labels, dim=1))  #convert logits to labels
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = stolen_model(data)
            loss = F.cross_entropy(output, target)

            #check for nan loss
            if torch.isnan(loss):
                print(f"Loss is nan at batch {batch_idx}")
                return  #stop training if nan is encountered

            loss.backward()
            
            #clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(stolen_model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')


In [9]:
#train stolen model
stolen_model = StolenModel().to(device)
print("Training the stolen model...")
train_stolen_model(stolen_model, device, train_data, train_labels, epochs=30)

#save stolen model
torch.save(stolen_model.state_dict(), 'stolen_model.pth')
print("Stolen model saved as 'stolen_model.pth'.")

Training the stolen model...
Epoch 1/30, Loss: 2.3033
Epoch 2/30, Loss: 2.2985
Epoch 3/30, Loss: 2.2931
Epoch 4/30, Loss: 2.2860
Epoch 5/30, Loss: 2.2764
Epoch 6/30, Loss: 2.2636
Epoch 7/30, Loss: 2.2464
Epoch 8/30, Loss: 2.2222
Epoch 9/30, Loss: 2.1862
Epoch 10/30, Loss: 2.1293
Epoch 11/30, Loss: 2.0334
Epoch 12/30, Loss: 1.8720
Epoch 13/30, Loss: 1.6399
Epoch 14/30, Loss: 1.3923
Epoch 15/30, Loss: 1.1805
Epoch 16/30, Loss: 1.0125
Epoch 17/30, Loss: 0.8841
Epoch 18/30, Loss: 0.7904
Epoch 19/30, Loss: 0.7208
Epoch 20/30, Loss: 0.6669
Epoch 21/30, Loss: 0.6229
Epoch 22/30, Loss: 0.5852
Epoch 23/30, Loss: 0.5525
Epoch 24/30, Loss: 0.5230
Epoch 25/30, Loss: 0.4962
Epoch 26/30, Loss: 0.4718
Epoch 27/30, Loss: 0.4493
Epoch 28/30, Loss: 0.4286
Epoch 29/30, Loss: 0.4098
Epoch 30/30, Loss: 0.3925
Stolen model saved as 'stolen_model.pth'.


Our final step is to evaluate the model.

In [10]:
#model evaluation
def evaluate_model(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  #we do not track gradients during eval
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            data_flat = data.view(data.size(0), -1) #input data flattening
            output = model(data_flat)

            #compute loss
            test_loss += F.cross_entropy(output, target, reduction='sum').item()  #sum batch loss
            
            #get prediction
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()  #compare with actual labels
            total += target.size(0)  #total number of samples

    #calculate avg loss & accuracy
    test_loss /= total
    accuracy = 100. * correct / total

    print(f'Average Test Loss: {test_loss:.4f}')
    print(f'Accuracy: {accuracy:.2f}% ({correct}/{total})')

In [11]:
#evaluation
#victim
print("\nEvaluating the Victim Model...")
evaluate_model(victim_model, device, test_loader)

#stolen
print("\nEvaluating the Stolen Model...")
evaluate_model(stolen_model, device, test_loader)


Evaluating the Victim Model...
Average Test Loss: 0.1785
Accuracy: 94.71% (9471/10000)

Evaluating the Stolen Model...
Average Test Loss: 0.4710
Accuracy: 86.23% (8623/10000)


From this result, we can see that our model's accuracy is 86.23%, which is quite low compared to the victim model's performance. To improve the surrogate model's performance, we can increase the number of epochs or change the general architecture (e.g. have more neurons in hidden layers or increase the number of hidden layers).

### The Task
Your task is to fit an MLP model on the CIFAR-10 dataset. You don't need to perform any attacks on it, just get a result. You can find the dataset [here](https://www.cs.toronto.edu/~kriz/cifar.html) (download the Python version). We will use the same libraries.

Reminder: you will need to set the batch size, the learning rate and the number of epochs outside your MLP class (these will passed as arguments). We also recommend splitting your class and function definitions, and data loading into separate cells and have a cell dedicated to training the model to save you time (you might need to re-run some parts of your code only rather than the whole script).

In [12]:
#your code goes here
#set batch size - set it to a value equal to 2^n
#e.g. 32, 64, 128, 256, etc
batch_size =

#set learning rate
learning_rate = 

#set epochs
epochs =

#obtain the CIFAR10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),  #convert images to PyTorch tensors
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  #normalise the data to [-1, 1]
])

train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)

test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)





SyntaxError: invalid syntax (2338102072.py, line 4)